# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [ ]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. ",
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. ",
        "A report valued the project at $3.2 billion.")

print(text)

('In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. ', 'He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. ', 'A report valued the project at $3.2 billion.')


## Q1

In [9]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
# - Then use nltk.sent_tokenize.
#
# Return: sentences (list of strings)

# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)
# TODO: apply sent_tokenize
marker = '<<<DOT>>>'
def protect_acronym_dots(s):
    return re.sub(r'\b((?:[A-Z]\.){2,})', lambda m: m.group(1).replace('.', marker), s)

def restore_acronym_dots(s):
    return s.replace(marker, '.')

text_protected = protect_acronym_dots(text)
sentences = sent_tokenize(text_protected)
sentences = [restore_acronym_dots(s) for s in sentences]

print('Sentences:')
for i, s in enumerate(sentences, 1):
    print(i, s)


Sentences:
1 In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.
2 He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q2

In [ ]:
# Q2 (1 pt): Regex normalization
# Convert: acronyms, meters->centimeters, $X.Y billion -> words
text_norm = text

def normalize_acronyms(s):
    # Remove dots in acronyms like U.P.C. -> UPC 
    return re.sub(r"\b((?:[A-Z]\\.){2,})", lambda m: m.group(1).replace('.', ''), s)

def meters_to_cm(s):
    return re.sub(r"(\d+(?:\.\d+)?)m\b", lambda m: f"{int(round(float(m.group(1))*100))} centimeters", s)

digit_map = {'0':'zero','1':'one','2':'two','3':'three','4':'four','5':'five','6':'six','7':'seven','8':'eight','9':'nine'}

def money_to_words(s):
    # Convert $3.2 billion -> three point two billion 
    def repl(m):
        num = m.group(1)
        words = []
        for ch in num:
            if ch == '.':
                words.append('point')
            elif ch.isdigit():
                words.append(digit_map[ch])
        return ' '.join(words) + ' billion'
    return re.sub(r"\$(\d+(?:\.\d+)?)\s*billion", repl, s, flags=re.IGNORECASE)

# Apply normalizations in sequence
text_norm = normalize_acronyms(text_norm)
text_norm = meters_to_cm(text_norm)
text_norm = money_to_words(text_norm)

print('Normalized text:')
print(text_norm)

Normalized text:
In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at three point two billion.


## Q3

In [ ]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
src = globals().get('text_norm', text)  
tokens = word_tokenize(src)
tags = nltk.pos_tag(tokens)
# Helper checks
def is_acronym(tok):
    return re.fullmatch(r'(?:[A-Z]\.?){2,}', tok) is not None
def is_mixedcase(tok):
    return any(c.islower() for c in tok) and any(c.isupper() for c in tok)
# Build new token list grouping consecutive NNP/NNPS into multiword proper nouns
new_tokens = []
i = 0
while i < len(tags):
    tok, tag = tags[i]
    if tag in ('NNP','NNPS'):
        # start group of proper nouns
        j = i
        group = []
        while j < len(tags) and tags[j][1] in ('NNP','NNPS'):
            group.append(tags[j][0])
            j += 1
        if len(group) > 1:
            # join multiword proper noun with underscore, preserve original casing
            new_tokens.append('_'.join(group))
        else:
            # single proper noun: preserve original casing (and acronyms uppercased)
            single = group[0]
            if is_acronym(single):
                new_tokens.append(re.sub(r'\.', '', single).upper())
            else:
                new_tokens.append(single)
        i = j
    else:
        # non-proper-noun token: apply rules
        if is_acronym(tok):
            new_tokens.append(re.sub(r'\.', '', tok).upper())
        elif is_mixedcase(tok):
            new_tokens.append(tok)  # keep MixedCase as-is
        else:
            new_tokens.append(tok.lower())
        i += 1

# Reconstruct text with simple punctuation handling
text_case = ''
for t in new_tokens:
    if re.fullmatch(r'[\.,;:\?!%\)\(\[\]\{\}]', t):
        text_case = text_case.rstrip() + t
    else:
        text_case = text_case + (' ' if text_case else '') + t

print('Text with case rules applied:')
print(text_case)

Text with case rules applied:
In mid-February 2026, the CEO of OpenAI, Sam_Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC. and UNESCO. A report valued the project at three point two billion.


## Q4

In [16]:
# Q4 (1 pt): Tokenization
src = globals().get('text_case', globals().get('text_norm', text))
tokens = word_tokenize(src)
print('Tokens:')
print(tokens)

Tokens:
['In', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', 'He', 'is', '186', 'centimeters', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'UPC', '.', 'and', 'UNESCO', '.', 'A', 'report', 'valued', 'the', 'project', 'at', 'three', 'point', 'two', 'billion', '.']


## Q5

In [ ]:
# Q5 (1 pt): Stopword removal
src_tokens = globals().get('tokens', word_tokenize(globals().get('text_case', globals().get('text_norm', text))))
stopset = set(stopwords.words('english'))
entities = {'OpenAI','Sam_Altman','Barcelona','UNESCO','UPC'}  
tokens_nostop = []
for t in src_tokens:
    # Keep entity tokens unchanged
    if t in entities:
        tokens_nostop.append(t)
        continue
    # Keep numbers and tokens with alphanumeric characters that are not stopwords
    if any(c.isalnum() for c in t):
        if t.lower() not in stopset:
            tokens_nostop.append(t)
    

print('Tokens without stopwords:')
print(tokens_nostop)

Tokens without stopwords:
['mid-February', '2026', 'CEO', 'OpenAI', 'Sam_Altman', 'visited', 'Barcelona', '186', 'centimeters', 'tall', 'met', 'researchers', 'UPC', 'UNESCO', 'report', 'valued', 'project', 'three', 'point', 'two', 'billion']


## Q6

In [ ]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

bigrams = None

# print(bigrams)


## Q7

In [ ]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = None
context_counts = None
model = None

def predict_next(prev_word, model, top_k=3):
    # TODO
    return None

# Example:
# print(predict_next("OpenAI", model, top_k=3))


## Q8

In [ ]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [ ]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = None
FP = None
FN = None
TN = None

accuracy = None
precision = None
recall = None
f1 = None

# print(accuracy, precision, recall, f1)
